# Imports

## Import Libraries

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tqdm import tqdm
import os
import matplotlib.pyplot as plt
import seaborn as sns
import math
from torch.utils.tensorboard import SummaryWriter 
import json

## Import Dataset Classes

In [4]:
from dataset_classes import ISO_NE, AT, BD_Dataset, ISO_NE_Small

## Import Models

In [1]:
from models import GLFN_TC_Attention, GLFN_TC_GlobalLocal, GLFN_TC_GraphGRU, GLFN_TC_Linear, GLFN_TC_MultiScale

## Import Training and Testing Loops

In [2]:
from helper_functions import train_model, test_model

# Main Function

In [ ]:
# Main Function
def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # # --- Load dataset (unscaled) ---
    # dataset = ISO_NE(
    #     csv_path="GLFN-TC\Datasets\ISO-NE\ISO-NE\selected_data_ISONE.csv",
    #     T_in=72,
    #     T_out=240,
    #     lag_hours=[24, 168],
    # )
    # --- Load dataset (unscaled) ---
    dataset = ISO_NE(
        csv_path="GLFN-TC\Datasets\ISO-NE\ISO-NE\selected_data_ISONE.csv",
        T_in=72,
        T_out=240,
        lag_hours=[1,2,6,12,24,168],
    )
    # --- Load dataset (unscaled) ---
    # dataset = BD_Dataset(
    #     csv_path="BangladeshData_2016_2024.csv",
    #     T_in=72,
    #     T_out=240,
    #     lag_hours=[1,2,6,12,24,168],
    # )
    # dataset = ISO_NE_Small(
    #     csv_path="ISO_NE_2013_2014.csv",
    #     T_in=72,
    #     T_out=240,
    #     lag_hours=[1,2,6,12,24,168],
    # )

    # --- 1. Define splits based on RAW data length ---
    total_len = len(dataset.df_numeric)
    train_split_idx = int(0.6 * total_len)
    val_split_idx = int(0.8 * total_len)
    
    print(f"Raw data length: {total_len}")
    print(f"Scaler 'train_size' (raw rows): {train_split_idx}")
    print(f"Scaler 'val_end' (raw rows): {val_split_idx}")

    # --- 2. Fit scaler only on the raw TRAIN subset ---
    scaler = StandardScaler()
    scaler.fit(dataset.df_numeric.iloc[:train_split_idx].values.astype(np.float32))

    # --- 3. Apply same scaler to all data ---
    dataset.apply_scaler(scaler)
    dataset.scaler = scaler  # keep for later inverse-transform

    # --- 4. Calculate SAMPLE indices to align with RAW splits ---
    
    # A sample 'idx' uses an input window from [idx] to [idx + T_in].
    # To prevent data leakage, the *input window* of a training sample
    # must not see any validation data. Validation data starts at `train_split_idx`.
    
    # The last training sample's input window must end at or before `train_split_idx - 1`.
    # idx + T_in - 1 <= train_split_idx - 1  =>  idx <= train_split_idx - T_in
    
    # The `range(start, end)` function's 'end' is exclusive, so this works perfectly.
    train_end = train_split_idx - dataset.T_in
    val_end = val_split_idx - dataset.T_in
    
    # Get the total number of *valid* samples
    effective_len = len(dataset)
    
    # Ensure our calculated indices don't exceed the total number of samples
    train_end = min(train_end, effective_len)
    val_end = min(val_end, effective_len)

    train_idx = range(0, train_end)
    val_idx = range(train_end, val_end)
    test_idx = range(val_end, effective_len)

    print(f"Total valid samples: {effective_len}")
    print(f"Train samples: {len(train_idx)}, Val samples: {len(val_idx)}, Test samples: {len(test_idx)}")
    
    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)
    test_subset = Subset(dataset, test_idx)

    hparams = {
        "N": dataset.N,
        "T_in": 72,
        "T_out": 240,
        "d": 32,
        "hidden_dim": 64,
        "GCN_Layer": 5,
        "dropout_forecast": 0.1,
        "dropout_gcn": 0.2,
        "dropout_temporal": 0.2,
        "lr": 1e-4,
        "scheduler_patience": 3,
        "batch_size": 64, # Updated from 16
        "epochs": 100,
        "weight_decay": 1e-4, # Added this new hyperparameter
    }
    
    # --- DataLoaders ---
    train_loader = DataLoader(train_subset, batch_size=hparams["batch_size"], shuffle=False)
    val_loader = DataLoader(val_subset, batch_size=hparams["batch_size"], shuffle=False)
    test_loader = DataLoader(test_subset, batch_size=hparams["batch_size"], shuffle=False)
    print(f"\n🚚 DataLoaders ready. Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")
    
    # --- Model ---
    model = GLFN_TC_Linear(
        N=hparams["N"],
        T_in=hparams["T_in"],
        T_out=hparams["T_out"],
        d=hparams["d"],
        hidden_dim=hparams["hidden_dim"],
        GCN_Layer=hparams["GCN_Layer"],
        dropout_gcn=hparams["dropout_gcn"],
        dropout_temporal=hparams["dropout_temporal"],
    ).to(device)
    
    # Run name
    run = "ISO_NE_Dataset_Main_LinearForecasting_Run1"
    
    
    log_dir = f"load/{run}"  # Define log directory for TensorBoard
    writer = SummaryWriter(log_dir)
    
    # Log hyperparameters as text (avoid add_hparams which requires a metric_dict)
    writer.add_text("hparams", json.dumps(hparams, indent=2))
    
    print("\n🚀 Training GLFN‑TC model on AT dataset...")
    model = train_model(
        model,
        train_loader,
        val_loader,
        epochs=hparams["epochs"],
        lr=hparams["lr"],
        device=device,
        scheduler_patience=hparams["scheduler_patience"],
        writer=writer,
        weight_decay=hparams["weight_decay"],
        save_path=f"Models/{run}_best_model.pth"
    )

    print("\n🧪 Testing model performance...\n")
    preds, trues = test_model(
        dataset=dataset,
        model=model, test_loader=test_loader,
        device=device,
        writer=writer
    )
    
    writer.close()

if __name__ == "__main__":
    main()

Loaded dataset with 18 features (target=demand), total rows=103608
Raw data length: 103608
Scaler 'train_size' (raw rows): 62164
Scaler 'val_end' (raw rows): 82886
Total valid samples: 103296
Train samples: 62092, Val samples: 20722, Test samples: 20482

🚚 DataLoaders ready. Train batches: 971, Val batches: 324, Test batches: 321

🚀 Training GLFN‑TC model on AT dataset...


c:\Users\khurs\Documents\GitHub\Load_Forecast_and_Balance\load_venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Epoch 1/100: 100%|██████████| 971/971 [00:21<00:00, 44.70it/s, batch_loss=0.946]


Epoch 001 | Train Loss: 1.0246 | Val Loss: 1.0947 | LR: 0.000100
✅ New best model saved (Val Loss: 1.094653)


Epoch 2/100:  23%|██▎       | 221/971 [00:04<00:16, 45.34it/s, batch_loss=0.687]


KeyboardInterrupt: 